# For geothermal Vienna basin, this notebook has been adjusted for the following: 
- Extract both finit and f0000 files, f0000 file is for initial temperature

# This notebook extracts realizations to be used for CMG simulations

* For CMG, set reverse_j = True when using generate_full_properties(). 
* The finit files must contain ACTID to identify active cells, in addition to properties such as PORO and PERMX.
* This notebook has been validated by (1) processing a realization using this notebook, (2) import processed realiztion into Petrel, (3) compare it with the realization in Petrel and found statistics are exactly the same. Note when exporting a property out of Petrel, setting undefined cells to be 0 (instead of -999) would generate the exact same file as a processed realization by this notebook.

## Step 1: Extract finit and f0000 files from simulation cases

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent
sys.path.append(str(repo_root / "src"))

from extract_filenames_files import extract_sorted_files

################# Start of user inputs ########## 
data_folder_path = repo_root/'data'/'Vienna_geothermal'/'sim_cases' # path to the folder containing simulation case subfolders, each with a .FINIT file
results_folder_path_finit = repo_root/'results'/'Vienna_geothermal'/'finit_files' # path to the folder where extracted .FINIT files will be copied. Will be created if it doesn't exist.
results_folder_path_f0000 = repo_root/'results'/'Vienna_geothermal'/'f0000_files' # path to the folder where extracted .F0000 files will be copied. Will be created if it doesn't exist.
################## End of user inputs  ##########

# Extract .FINIT files from the data folder and copy them to the results folder
extract_sorted_files(
        folder_dir = data_folder_path,
        save_dir = results_folder_path_finit,
        file_extension = ['.FINIT'],
        show_summary = True,
        show_filenames = False
)
# Extract .F0000 files from the data folder and copy them to the results folder
extract_sorted_files(
        folder_dir = data_folder_path,
        save_dir = results_folder_path_f0000,
        file_extension = ['.F0000'],
        show_summary = True,
        show_filenames = False
)

Extracting files: 3it [00:00, 10.26it/s]


Total files copied (.FINIT): 2


Extracting files: 3it [00:00, 68.14it/s]

Total files copied (.F0000): 2


## Step 2: Extract properties

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent
sys.path.append(str(repo_root / "src"))

from extract_properties_from_finit import extract_properties_from_finit
from generate_full_properties import generate_full_properties
from CMG_format_compress import CMG_format_compress
from extract_filenames_files import extract_sorted_filenames

################# Start of user inputs ########## 
finit_folder_path = repo_root/'results'/'Vienna_geothermal'/'finit_files' # path to the folder containing .FINIT files extracted from simulation cases
f0000_folder_path = repo_root/'results'/'Vienna_geothermal'/'f0000_files' # path to the folder containing .F0000 files extracted from simulation cases
save_folder_path = repo_root/'results'/'Vienna_geothermal'/'properties' # path to the folder where extracted properties will be saved. Will be created if it doesn't exist.
grid_shape = (139, 248, 23) # shape of the geomodel grid (i, j, k)
finit_property_list = ['PORO', 'PERMX']  # list of property keywords to extract from .FINIT files
f0000_property_list = ['TEMP']  # list of property keywords to extract from .F0000 files
################## End of user inputs  ##########

save_folder_path.mkdir(exist_ok=True)

# extract finit file names from its folder
finit_file_names = extract_sorted_filenames(
        folder_dir = finit_folder_path,
        is_save = False,
        save_dir = None,
        save_name = None
)
# extract f0000 file names from its folder
f0000_file_names = extract_sorted_filenames(
        folder_dir = f0000_folder_path,
        is_save = False,
        save_dir = None,
        save_name = None
)
# guard: both folders must have the same number of files so zip() pairs them correctly
assert len(finit_file_names) == len(f0000_file_names), \
    f"{len(finit_file_names)} finit vs {len(f0000_file_names)} f0000 files"

# extract properties
for finit_file_name, f0000_file_name in zip(finit_file_names, f0000_file_names):
    finit_file_path = finit_folder_path / finit_file_name
    f0000_file_path = f0000_folder_path / f0000_file_name
    # some finit files contains PORO cells < active cells, so use try-except-continue to avoid interuption
    try:
        # STEP 1: Extract properties from FINIT file for active cells only
        extracted_property_dict = extract_properties_from_finit(
            finit_file_path = finit_file_path,
            keywords = finit_property_list + ['ACTID'],
            is_save = False,
            save_dir = save_folder_path,
            save_name = finit_file_name.split('.')[0],
            show_summary = False
        )
        # Extract properties from F0000 file for active cells only
        extracted_property_dict_F0000 = extract_properties_from_finit(
            finit_file_path = f0000_file_path,
            keywords = f0000_property_list,
            is_save = False,
            save_dir = save_folder_path,
            save_name = f0000_file_name.split('.')[0],
            show_summary = False
        )
        # Add the temperature property from the F0000 file to the F0000 property dictionary
        extracted_property_dict ['TEMP'] = extracted_property_dict_F0000['TEMP']
        
        # STEP 2: Generate full properties for all cells (fill inactive cells with zeros)
        full_property_dict = generate_full_properties(
            property_dict = extracted_property_dict,
            property_list = finit_property_list + f0000_property_list, 
            grid_shape = grid_shape,
            is_save = False,
            save_dir = save_folder_path,
            save_name = finit_file_name.split('.')[0],
            show_summary = False,
            reverse_j = True
            )


        # STEP 3: Compress full properties to CMG format (repeated values as N*value)
        for key in finit_property_list + f0000_property_list:
            CMG_format_compress(
                array = full_property_dict[key], 
                keyword = key, 
                max_line_length = 80,
                show_summary = False,
                save_dir = save_folder_path,
                save_name = finit_file_name.split('.')[0]
            )
    except (ValueError, KeyError) as e:
        print(f"Warning: Skipping '{finit_file_name}'. Reason: {e}")
        continue


Saved compressed PORO data to: /Users/jijuid/Library/CloudStorage/GoogleDrive-jihuid@stanford.edu/My Drive/Github/preCMGsim/results/Vienna_geothermal/properties/JD_GEOTHERMAL_1_PORO.dat
Saved compressed PERMX data to: /Users/jijuid/Library/CloudStorage/GoogleDrive-jihuid@stanford.edu/My Drive/Github/preCMGsim/results/Vienna_geothermal/properties/JD_GEOTHERMAL_1_PERMX.dat
Saved compressed TEMP data to: /Users/jijuid/Library/CloudStorage/GoogleDrive-jihuid@stanford.edu/My Drive/Github/preCMGsim/results/Vienna_geothermal/properties/JD_GEOTHERMAL_1_TEMP.dat
Saved compressed PORO data to: /Users/jijuid/Library/CloudStorage/GoogleDrive-jihuid@stanford.edu/My Drive/Github/preCMGsim/results/Vienna_geothermal/properties/JD_GEOTHERMAL_2_PORO.dat
Saved compressed PERMX data to: /Users/jijuid/Library/CloudStorage/GoogleDrive-jihuid@stanford.edu/My Drive/Github/preCMGsim/results/Vienna_geothermal/properties/JD_GEOTHERMAL_2_PERMX.dat
Saved compressed TEMP data to: /Users/jijuid/Library/CloudStorage/

## Step 3: Collect property file names to be used for LSH sampling

In [1]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent
sys.path.append(str(repo_root / "src"))

from extract_filenames_files import extract_sorted_filenames

################# Start of user inputs ########## 
property_filenames = extract_sorted_filenames(
        folder_dir = repo_root/'results'/'Vienna_geothermal'/'properties', # path to the folder containing the extracted properties
        is_save = True, # whether to save the filenames to a CSV file
        save_dir = repo_root/'results'/'Vienna_geothermal'/'property_file_names', # path to the folder where the filenames will be saved
        save_name = 'test_geothermal_filenames.csv', # name of the CSV file where the filenames will be saved
        show_filenames = True # whether to print the filenames to the console
)
################# End of user inputs ############ 

Extracted and sorted filenames:
JD_GEOTHERMAL_1_PERMX.dat
JD_GEOTHERMAL_1_TEMP.dat
JD_GEOTHERMAL_1_PORO.dat
JD_GEOTHERMAL_2_PERMX.dat
JD_GEOTHERMAL_2_PORO.dat
JD_GEOTHERMAL_2_TEMP.dat
